# Ch.4 — Neural Collaborative Filtering

> **The story.** In **2017**, Xiangnan He and colleagues at the National University of Singapore published "Neural Collaborative Filtering" (_WWW 2017_), arguing that the inner product in matrix factorisation is _too simple_ to capture complex user–item interactions. Their key insight: **replace the dot product with a neural network** that takes user and item embeddings as input and learns an arbitrary interaction function. The architecture — called **NeuMF** — combines two parallel paths: a **Generalised Matrix Factorisation (GMF)** path for linear interactions and a **Multi-Layer Perceptron (MLP)** path for non-linear ones, fused in a final prediction layer. GMF and MLP use _separate_ embedding spaces, letting each path specialise. The paper showed consistent improvements over MF on MovieLens and Pinterest, and launched a wave of deep-learning recommenders at Alibaba, JD.com, and Pinterest. Every "deep collaborative filter" you encounter today traces its lineage to this six-page paper.
>
> **Where you are in the curriculum.** Chapter four of the FlixAI track. Matrix factorisation (Ch.3) achieved **78% HR@10** on MovieLens 100k but is limited to linear interactions ($\hat{r} = \mathbf{u}^\top\mathbf{v}$). Neural CF replaces the dot product with a _learnable_ non-linear function, capturing taste patterns like "loves sci-fi and comedy separately but hates sci-fi comedy hybrids" — interactions the dot product cannot express. This is the **first deep-learning model** in the Recommender Systems track.
>
> **Notation.** $\mathbf{p}_u^G, \mathbf{q}_i^G$ — user/item embeddings in the **GMF** space ($\in \mathbb{R}^d$); $\mathbf{p}_u^M, \mathbf{q}_i^M$ — user/item embeddings in the **MLP** space; $\odot$ — element-wise (Hadamard) product; $\oplus$ — concatenation; $\hat{y}_{ui}$ — predicted interaction probability $\in [0,1]$; $\mathcal{Y}^+, \mathcal{Y}^-$ — positive/negative pairs; $k$ — negative-sampling ratio (default 4).

---

## §0 · The Challenge — Where We Are

> **The mission**: Launch **FlixAI** — >85% HR@10 across 5 constraints: (1) ACCURACY >85%, (2) COLD START, (3) SCALABILITY <200ms, (4) DIVERSITY, (5) EXPLAINABILITY "Because you liked X."

**Progress so far:** Ch.1 → 42%. Ch.2 → 65%. Ch.3 MF → 78%. **Still 7 points short.** MF plateaus at 78% even after increasing factors ($d=8 \to 16 \to 32$, +0.5% each). The architecture itself is the bottleneck: the dot product $\mathbf{u}^\top\mathbf{v}$ is fundamentally linear and cannot encode "likes A and B separately but hates A+B together." Replace it with an MLP that can learn any interaction function.

```mermaid
flowchart LR
 MF["Ch.3: MF\nHR@10 = 78%\nlinear dot product"] --> EMB["Separate GMF\n+ MLP Embeddings"]
 EMB --> GMF["GMF Path\n(linear ⊙)"]
 EMB --> MLP["MLP Path\n(non-linear ReLU)"]
 GMF --> FUSE["Fuse & Predict\nσ(w·GMF ⊕ w·MLP)"]
 MLP --> FUSE
 FUSE --> EVAL["NeuMF\nHR@10 ≈ 82%"]

 style MF fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style EMB fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style FUSE fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style EVAL fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Dataset:** MovieLens 100k | **Task:** Build NeuMF (Neural Matrix Factorization) with PyTorch | **Outcome:** NCF = ~82% HR@10


## §1 · The Core Idea

Matrix factorization predicts ratings with a dot product: if your "sci-fi dimension" is 0.9 and a movie's "sci-fi dimension" is 0.8, you'd rate it highly. But taste is not additive. Consider a user who loves both sci-fi and comedy separately but **hates** sci-fi comedies — the dot product treats dimensions independently and can never capture this cross-dimension interaction.

**Neural CF's answer:** replace the dot product with a neural network. Given user and item as input, an MLP can learn any function of their embeddings — including cross-dimension interactions. The NeuMF architecture uses **two parallel paths** so each can specialise:

- **GMF path** (element-wise product $\odot$): captures linear, dimension-by-dimension alignment — the same signal as MF
- **MLP path** (concatenate → dense layers): captures non-linear cross-dimension interactions — "likes A and B separately but hates A+B"

Both paths use _separate_ embedding spaces (the GMF embeddings are not shared with the MLP), letting each specialise. Their outputs are concatenated and passed through a final sigmoid layer.

```mermaid
flowchart LR
    UG["User\nGMF emb p_u^G"] --> GMFOP["⊙ element-wise\nproduct"]
    IG["Item\nGMF emb q_i^G"] --> GMFOP
    UM["User\nMLP emb p_u^M"] --> CONCAT["⊕ concatenate\n→ MLP layers\n(ReLU)"]
    IM["Item\nMLP emb q_i^M"] --> CONCAT
    GMFOP --> FUSE["⊕ fuse → σ(w·x)\nfinal prediction"]
    CONCAT --> FUSE
    FUSE --> OUT["ŷ_ui ∈ [0,1]\ninteraction score"]

    style UG fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style IG fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style UM fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style IM fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style GMFOP fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style CONCAT fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style FUSE fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style OUT fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Trained on **implicit feedback** (binary: interacted or not) with **negative sampling** — for each observed interaction, sample $k$ items the user _didn't_ interact with as negatives.

> **Optional depth:** $\hat{y}_{ui} = \sigma\!\left(\mathbf{h}^T\!\left[\mathbf{p}_u^G \odot \mathbf{q}_i^G \;\oplus\; \text{MLP}(\mathbf{p}_u^M \oplus \mathbf{q}_i^M)\right]\right)$ — where $\mathbf{h}$ is the learnable fusion weight vector, $\odot$ is element-wise product, and $\oplus$ is concatenation.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Libraries loaded.")

In [ ]:
# ── Load MovieLens 100k ───────────────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"]
)

n_users = ratings["user_id"].max()
n_items = ratings["item_id"].max()

# Leave-one-out split
ratings_sorted = ratings.sort_values("timestamp")
test = ratings_sorted.groupby("user_id").tail(1).copy()
train = ratings_sorted.drop(test.index).copy()

# Build set of items each user rated (for negative sampling)
user_rated = train.groupby("user_id")["item_id"].apply(set).to_dict()
all_items = set(range(1, n_items + 1))

print(f"Users: {n_users}  Items: {n_items}")
print(f"Train: {len(train):,}  Test: {len(test):,}")

In [ ]:
# ── Dataset with Negative Sampling ────────────────────────────────────────
class NCFDataset(Dataset):
    """Each positive (user, item) gets k negative samples."""

    def __init__(self, train_df, n_items, user_rated, n_neg=4):
        self.users = train_df["user_id"].values
        self.items = train_df["item_id"].values
        self.n_items = n_items
        self.user_rated = user_rated
        self.n_neg = n_neg

    def __len__(self):
        return len(self.users) * (1 + self.n_neg)

    def __getitem__(self, idx):
        base_idx = idx // (1 + self.n_neg)
        sub_idx = idx % (1 + self.n_neg)

        user = self.users[base_idx]
        if sub_idx == 0:
            # Positive sample
            return user, self.items[base_idx], 1.0
        else:
            # Negative sample
            neg_item = np.random.randint(1, self.n_items + 1)
            while neg_item in self.user_rated.get(user, set()):
                neg_item = np.random.randint(1, self.n_items + 1)
            return user, neg_item, 0.0


train_dataset = NCFDataset(train, n_items, user_rated, n_neg=4)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)

print(f"Dataset size: {len(train_dataset):,} (1 pos + 4 neg per interaction)")

In [ ]:
# ── NeuMF Model ───────────────────────────────────────────────────────────
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, d_gmf=16, d_mlp=32, mlp_layers=[64, 32, 16]):
        super().__init__()
        # GMF embeddings
        self.user_gmf = nn.Embedding(n_users + 1, d_gmf, padding_idx=0)
        self.item_gmf = nn.Embedding(n_items + 1, d_gmf, padding_idx=0)
        # MLP embeddings (separate from GMF)
        self.user_mlp = nn.Embedding(n_users + 1, d_mlp, padding_idx=0)
        self.item_mlp = nn.Embedding(n_items + 1, d_mlp, padding_idx=0)

        # MLP tower
        layers = []
        input_dim = d_mlp * 2
        for hidden in mlp_layers:
            layers.append(nn.Linear(input_dim, hidden))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            input_dim = hidden
        self.mlp = nn.Sequential(*layers)

        # Fusion layer
        self.output = nn.Linear(d_gmf + mlp_layers[-1], 1)
        self.sigmoid = nn.Sigmoid()

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, user_ids, item_ids):
        # GMF path: element-wise product
        gmf = self.user_gmf(user_ids) * self.item_gmf(item_ids)

        # MLP path: concat → dense layers
        mlp_input = torch.cat(
            [self.user_mlp(user_ids), self.item_mlp(item_ids)], dim=-1
        )
        mlp_out = self.mlp(mlp_input)

        # Fuse and predict
        fused = torch.cat([gmf, mlp_out], dim=-1)
        return self.sigmoid(self.output(fused)).squeeze(-1)


model = NeuMF(n_users, n_items, d_gmf=16, d_mlp=32, mlp_layers=[64, 32, 16]).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

n_epochs = 15
train_losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    for users, items, labels in train_loader:
        users = users.long().to(device)
        items = items.long().to(device)
        labels = labels.float().to(device)

        preds = model(users, items)
        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    print(f"  Epoch {epoch+1:>2d}/{n_epochs}: Loss = {avg_loss:.4f}")

print("Training complete.")

In [ ]:
# ── Training Loss Curve ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(train_losses) + 1), train_losses, color="#8e44ad", linewidth=2)
ax.set(xlabel="Epoch", ylabel="BCE Loss", title="NeuMF — Training Convergence")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/ncf_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Evaluate: HR@10 and NDCG@10 ──────────────────────────────────────────
model.eval()

top_k_ncf = {}
with torch.no_grad():
    for _, row in test.iterrows():
        user_id = int(row["user_id"])
        rated = user_rated.get(user_id, set())

        # Score all items
        user_tensor = torch.tensor([user_id] * (n_items + 1), dtype=torch.long).to(
            device
        )
        item_tensor = torch.arange(0, n_items + 1, dtype=torch.long).to(device)
        scores = model(user_tensor, item_tensor).cpu().numpy()

        # Mask rated items
        for r in rated:
            scores[r] = -np.inf
        scores[0] = -np.inf

        top_k_ncf[user_id] = np.argsort(scores)[-10:][::-1].tolist()


def hit_rate_at_k(test_df, top_k_per_user, k=10):
    hits = 0
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            hits += 1
    return hits / len(test_df)


def ndcg_at_k(test_df, top_k_per_user, k=10):
    ndcgs = []
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            rank = recs.index(test_item) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)


hr_ncf = hit_rate_at_k(test, top_k_ncf, k=10)
ndcg_ncf = ndcg_at_k(test, top_k_ncf, k=10)

print(f"NeuMF Results:")
print(f"  HR@10   = {hr_ncf:.3f} ({hr_ncf*100:.1f}%)")
print(f"  NDCG@10 = {ndcg_ncf:.4f}")

In [ ]:
# ── Compare All Methods So Far ────────────────────────────────────────────
results = pd.DataFrame(
    {
        "Method": ["Popularity", "Item-CF", "MF (d=20)", "NeuMF"],
        "HR@10": [0.42, 0.68, 0.78, hr_ncf],
    }
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#95a5a6", "#e67e22", "#27ae60", "#8e44ad"]
bars = ax.bar(
    results["Method"], results["HR@10"] * 100, color=colors, edgecolor="white"
)
ax.axhline(85, color="red", linestyle="--", alpha=0.7, label="Target: 85%", linewidth=2)
ax.set(ylabel="Hit Rate@10 (%)", title="FlixAI Progress — All Methods Compared")
ax.legend(fontsize=12)

for bar, val in zip(bars, results["HR@10"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{val*100:.0f}%",
        ha="center",
        fontsize=12,
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig("img/ncf_all_methods.png", dpi=150, bbox_inches="tight")
plt.show()

### What §1 established — and what it still doesn't solve

[Done] **Non-linear interactions learned.** The MLP path captures cross-dimension taste interactions the dot product cannot encode.

[Done] **HR@10 lifted.** From ~78% (Ch.3 MF) to ~82% — closing the gap to the 85% target by learning richer representations.

[Done] **Implicit feedback.** Trained on binary interaction signals, not just explicit ratings — more applicable to real systems.

**Still open:**

- **Cold start:** New users have no embedding in the model. Without at least one interaction, NeuMF cannot score anything. NCF makes this _worse_ than MF — each embedding is now task-specific and harder to initialise from content.
- **Content blindness:** The model treats every movie as an opaque integer ID. It does not know that _Dune_ is sci-fi directed by Denis Villeneuve. A new movie with 3 ratings has a near-random embedding.
- **3 points remaining:** 82% HR@10 is 3 points from the 85% target. The gap is structural — NCF cannot see item content. Ch.5 closes it.


In [ ]:
# ── Embedding Visualisation ───────────────────────────────────────────────
from sklearn.decomposition import PCA

# Extract GMF item embeddings
item_emb = model.item_gmf.weight.data.cpu().numpy()[1:]  # skip padding

pca = PCA(n_components=2, random_state=SEED)
emb_2d = pca.fit_transform(item_emb)

movies_df = pd.read_csv(
    url + "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    names=[
        "item_id",
        "title",
        "release_date",
        "video_release",
        "url",
        "unknown",
        "Action",
        "Adventure",
        "Animation",
        "Children",
        "Comedy",
        "Crime",
        "Documentary",
        "Drama",
        "Fantasy",
        "Film-Noir",
        "Horror",
        "Musical",
        "Mystery",
        "Romance",
        "Sci-Fi",
        "Thriller",
        "War",
        "Western",
    ],
    usecols=range(24),
)

genre_cols = ["Action", "Comedy", "Drama", "Horror", "Sci-Fi", "Romance"]
dominant_genre = movies_df[genre_cols].idxmax(axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
for genre in genre_cols:
    mask = dominant_genre == genre
    ax.scatter(
        emb_2d[mask.values[: len(emb_2d)], 0],
        emb_2d[mask.values[: len(emb_2d)], 1],
        alpha=0.4,
        s=15,
        label=genre,
    )

ax.set(
    xlabel="PC 1",
    ylabel="PC 2",
    title="NeuMF GMF Embeddings — PCA Projection (Coloured by Genre)",
)
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig("img/ncf_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()

## Progress Check

**Checkpoint:** FlixAI — hit@10 advanced from ~78% to ~82%, closing within 3 points of the >85% target in this chapter. Non-linear embeddings capture taste interactions that linear matrix factorization cannot.

| #   | Constraint     | Target                | Ch.4 Status                                 |
| --- | -------------- | --------------------- | ------------------------------------------- |
| 1   | ACCURACY       | >85% HR@10            | ~82% — 3 points remaining                   |
| 2   | COLD START     | New users/items       | [No] Embeddings require interaction history |
| 3   | SCALABILITY    | 1M+ ratings           | GPU helpful for training                    |
| 4   | DIVERSITY      | Not just popular      | Richer embeddings help                      |
| 5   | EXPLAINABILITY | "Because you liked X" | Neural network = black box                  |

**Bottom line**: 82% hit rate — just 3 points from target. Non-linear MLP captures taste interactions linear MF missed. But the model is content-blind — it cannot handle new items or reason about genres.

**Next**: Ch.5 — Hybrid Systems → add content features (genres, demographics) to close the final 3-point gap.


## Exercises

**Exercise 1 — GMF-Only vs MLP-Only**
Train GMF-only and MLP-only models separately. Compare HR@10 against the full NeuMF. Which component contributes more?

**Exercise 2 — Negative Sampling Ratio**
Train NeuMF with k=1, 4, and 10 negatives per positive. How does the ratio affect HR@10 and training time?

**Exercise 3 — Pre-Training**
Pre-train GMF and MLP separately, then initialise NeuMF with their weights and fine-tune. Does pre-training improve HR@10?


In [ ]:
# ── Exercise 1 scaffold — GMF-Only vs MLP-Only ───────────────────────────
# TODO: Create GMF-only model (no MLP path) and MLP-only model (no GMF path)
# Compare HR@10

# class GMFOnly(nn.Module):
#     def __init__(self, n_users, n_items, d=16):
#         ...  # Only GMF path
#     def forward(self, user_ids, item_ids):
#         gmf = self.user_emb(user_ids) * self.item_emb(item_ids)
#         return self.sigmoid(self.output(gmf)).squeeze(-1)

pass

In [ ]:
# ── Exercise 2 scaffold — Negative Sampling Ratio ────────────────────────
# TODO: Train with n_neg=1, 4, 10 and compare

# for n_neg in [1, 4, 10]:
#     dataset = NCFDataset(train, n_items, user_rated, n_neg=n_neg)
#     loader = DataLoader(dataset, batch_size=256, shuffle=True)
#     model = NeuMF(n_users, n_items)
#     # ... train, evaluate HR@10

pass

In [ ]:
# ── Exercise 3 scaffold — Pre-Training ───────────────────────────────────
# TODO: Pre-train GMF and MLP separately, then:
# 1. Copy GMF embeddings into NeuMF's GMF path
# 2. Copy MLP embeddings + weights into NeuMF's MLP path
# 3. Fine-tune NeuMF with smaller learning rate

# gmf_model = GMFOnly(n_users, n_items)
# mlp_model = MLPOnly(n_users, n_items)
# ... train both, then transfer weights to NeuMF

pass